# PaperQA2 - 30 Negative-Control Questions

Runs ONLY the PaperQA2 arm on the 30 negative-control questions, checkpointing
each answer. Split out of notebook 11 so this heavy arm can be run and
debugged on its own.

Setup is identical to notebook 6 (t21 embedder, evidence_k=12, max 8 sources)
and reuses the cached index paperqa_docs_2026-09-02_articles.pkl that notebook
6 built - the corpus is NOT re-indexed here. Answers are written to the SAME
checkpoint file notebook 15 Cell 3 reads
(paperqa_answers_negative_2026-09-02_articles.json in the
negative_control_refusals output dir), so after this notebook finishes,
notebook 15 reads them directly.

RUN WITH THE paperqa KERNEL (the paperqa conda env - Kernel > Change Kernel >
paperqa), the same env the positive-control PaperQA2 benchmark (notebook 6)
ran in. Resume-safe: re-running the answer cell skips questions that already
have an answer. PaperQA2's stock decline is "I cannot answer.".

## Cell 1 - Configuration

In [ ]:
# ---- Cell 1: configuration ----
import json, os, re, sys, time
from pathlib import Path

import pandas as pd
import requests

NB_DIR   = Path.cwd()
ANALYSIS = (NB_DIR.parent if NB_DIR.name == 'nbs' else NB_DIR).resolve()
BENCH    = ANALYSIS / 'benchmarks'
OUT_DIR  = ANALYSIS / 'output' / 'ScientificReportRevisionV1' / 'negative_control_refusals'
OUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS_XLSX = ANALYSIS / 'input' / 'Revised_DownSyndrome_60_Questions_Final_linked.xlsx'
sys.path.insert(0, str(BENCH))

PQA_CORPUS_DATE = '2026-09-02_articles'
PQA_CORPUS      = BENCH / f'paperqa_corpus_{PQA_CORPUS_DATE}'
PQA_OUT         = ANALYSIS / 'output' / 'paperqa_benchmark'
PQA_DOCS_PKL    = PQA_OUT / f'paperqa_docs_{PQA_CORPUS_DATE}.pkl'
PQA_EVIDENCE_K, PQA_MAX_SOURCES, PQA_EMBED = 12, 8, 't21'
# Same checkpoint path notebook 15 Cell 3 uses - do not change one without the other
ANS_PQA = OUT_DIR / f'paperqa_answers_negative_{PQA_CORPUS_DATE}.json'
assert PQA_DOCS_PKL.exists(), (
    f'cached PaperQA2 index missing: {PQA_DOCS_PKL} - run notebook 6 Cell 3 first')
print('questions  :', QUESTIONS_XLSX.name, '| exists:', QUESTIONS_XLSX.exists())
print('checkpoint :', ANS_PQA)
print('index      :', PQA_DOCS_PKL.name, f'({PQA_DOCS_PKL.stat().st_size/1e6:.0f} MB)')


## Cell 2 - The 30 negative-control questions

In [ ]:
# ---- Cell 2: the 30 negative-control questions ----
df_q = pd.read_excel(QUESTIONS_XLSX, sheet_name='Questions_for_testing')
df_q.columns = [c.strip() for c in df_q.columns]
neg = df_q[df_q['CTRL_Type'].str.strip().str.lower() == 'negative'].copy().reset_index(drop=True)
neg = neg.rename(columns={'Original question': 'question'})
neg['question'] = neg['question'].astype(str).str.strip()
neg['sl_no'] = neg['SLNo'].astype(str).str.strip().str.replace(r'^[Qq]', '', regex=True)
assert len(neg) == 30, f'expected 30 negative controls, found {len(neg)}'
print(f'negative controls: {len(neg)}')
neg[['sl_no', 'question']]


## Cell 3 - Load settings and the cached index

In [ ]:
# ---- Cell 3: settings + cached index ----
import pickle
from paperqa_adapter import make_settings, query_docs, answer_text

settings = make_settings(paper_directory=PQA_CORPUS,
                         index_directory=PQA_OUT / 'pqa_index',
                         evidence_k=PQA_EVIDENCE_K,
                         answer_max_sources=PQA_MAX_SOURCES,
                         embed_provider=PQA_EMBED)
print('llm       :', settings.llm)
print('summary   :', settings.summary_llm)
print('embedding :', settings.embedding)

t0 = time.time()
docs = pickle.loads(PQA_DOCS_PKL.read_bytes())
print(f'loaded cached index in {time.time()-t0:.0f}s '
      f'({len(getattr(docs, "docs", {}))} documents)')


## Cell 4 - Run the 30 questions (checkpointed, resume-safe)

In [ ]:
# ---- Cell 4: PaperQA2 answers ----
answers = json.loads(ANS_PQA.read_text()) if ANS_PQA.exists() else {}

for i, row in neg.iterrows():
    key, q = row['sl_no'], row['question']
    if answers.get(key, {}).get('answer'):
        print(f'  [{i+1}/30] [RESUME] SLNo {key}')
        continue
    t0 = time.time()
    try:
        resp = query_docs(docs, q, settings)
        sess = getattr(resp, 'session', resp)
        ctxs = getattr(sess, 'contexts', []) or []
        answers[key] = {'question': q, 'answer': answer_text(resp),
                        'secs': round(time.time() - t0, 1),
                        'n_contexts': len(ctxs)}
    except Exception as exc:
        answers[key] = {'question': q, 'answer': '', 'error': str(exc)[:200]}
        print(f'  [{key}] ERROR {str(exc)[:110]}')
    ANS_PQA.write_text(json.dumps(answers, indent=1, ensure_ascii=False))
    print(f'  [{i+1}/30] {answers[key].get("secs", "-")}s  {q[:62]}')

neg['paperqa_answer'] = [answers.get(s, {}).get('answer', '') for s in neg['sl_no']]
n_ok = int((neg['paperqa_answer'].str.len() > 0).sum())
print(f'\nanswered: {n_ok}/30')
if n_ok:
    print(f'mean answer length: {neg.loc[neg["paperqa_answer"].str.len() > 0, "paperqa_answer"].str.split().str.len().mean():.0f} words')
if n_ok < 30:
    print('re-run this cell to retry the failed/missing questions')


---

Output: paperqa_answers_negative_2026-09-02_articles.json in
analysis/output/ScientificReportRevisionV1/negative_control_refusals/.
When all 30 are answered, run notebook 15 - its Cell 3 loads these answers
from the checkpoint and the classification/Excel/figure cells do the rest.